# 13 · Clip polygons, checked against shapely

## What this checks

**Clip polygons** cuts a set of areas to a boundary. It keeps the part of each area inside the boundary. An area that
falls wholly outside is dropped, not returned empty. This check covers release 0.5.0.

An area that is not cut claims ground the report does not cover. So it is worth checking.

## How it is checked

We cut the same areas with **shapely**, Python's link to GEOS, the geometry engine much GIS software uses. Fieldwork
uses polyclip-ts, a geometry library for the browser. Nothing here uses Fieldwork's code.

Both sides work on longitude and latitude directly. That is what the widget declares: it does not project first. So
the check compares what the widget claims, not what it might have claimed.

The data come from the John Snow worked example. They are the nearest-pump areas, called Voronoi cells, cut to the
rectangle the report covers. The cells are built inside a larger frame and reach past the rectangle. So the cut does
real work.

## What should happen

1. **The same areas survive the cut.** An area that falls wholly outside is dropped. So a disagreement about whether
   an area is empty shows up as a different count.
2. **Each cut area matches shapely's**, with at most 1 × 10⁻¹³ square degrees of ground in one but not the other.
   Areas are matched by their own names.
3. **Nothing escapes the boundary.** Every area in the output lies within it.

The tolerance is in square degrees, because the operation works in degrees. At this latitude, 1 × 10⁻¹³ square
degrees is less than a square millimetre on the ground. These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["shapely", "pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_13_clip_polygons import run

report = run(Path.cwd() / "fixtures" / "clip-polygons-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"])
print(f"polygons in            {report['polygonsIn']}")
print(f"survived, fieldwork    {report['polygonsOut']}")
print(f"survived, shapely      {report['referenceOut']}")
print(f"worst symmetric difference  {report['worstSymmetricDifferenceSquareDegrees']:.3e} square degrees")
print(f"polygons escaping the boundary  {report['escapedTheBoundary'] or 'none'}")
print()
for row in report["perPolygon"]:
    print(f"  {row['id']:<12} {row['symmetricDifferenceSquareDegrees']:.3e}")

## The result on record

First recorded on 11 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.5.0:
**Agrees**.

All 8 of the 8 areas survived the cut on both sides. The largest mismatch is about 6 × 10⁻¹⁸ square degrees, far
inside the tolerance. No area escaped the boundary.

Running the cell above repeats the check on the same data.

## What this does not show

- That cutting to a reporting rectangle is the right thing to do. A cut catchment is the part of that catchment the
  report covers. That is a different thing from the catchment, and both sides produce it the same way.
- That cutting in degrees is safe for large areas. It was measured only on this rectangle, about 1.4 km across.